# The Arabidopsis microarray corpus in OSDR

Before RNA-seq became routine, the space-biology community characterised the
*Arabidopsis thaliana* spaceflight response almost entirely with **DNA
microarrays**. Those datasets are still archived in OSDR, still reprocessed
through the standard GeneLab pipeline, and — because they share a common
platform and a common analysis — they can be read *together* rather than one
at a time.

This chapter does three things:

1. **Assembles the corpus** — every Arabidopsis dataset in OSDR whose assay is a
   DNA microarray — and summarises what the studies collectively set out to test.
2. **Compares how they were run** — tissue, growth hardware, plant age, light
   regime and the applied stimulus — straight from the archived metadata.
3. **Finds the loci** whose expression variance tracks each study's *own* factor,
   then asks which loci recur across independent spaceflight experiments.

> Sections 1–2 query the [OSDR biodata API](https://visualization.osdr.nasa.gov/biodata/api/)
> live. Section 3 uses a pre-computed differential-expression table (the raw
> GeneLab DE files are 20–800 MB each); the provenance is documented where it is used.


In [1]:
import json, warnings, io
import requests
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = 'notebook_connected'
warnings.filterwarnings('ignore')

API = 'https://visualization.osdr.nasa.gov/biodata/api/v2'
PLANT_GREEN = '#2ca02c'


## 1. Assembling the corpus

The OSDR metadata API lets us pull the organism and assay technology for every
sample in the repository, then keep the accessions where *Arabidopsis thaliana*
meets *DNA microarray*.


In [2]:
def find_arabidopsis_microarray():
    """Live query: OSDR accessions that are Arabidopsis AND DNA microarray."""
    params = {
        'study.characteristics.organism': '',
        'investigation.study assays.study assay technology type': '',
    }
    r = requests.get(API + '/query/metadata/', params=params, timeout=120)
    df = pd.read_csv(io.StringIO(r.text), low_memory=False)
    org = 'study.characteristics.organism'
    tech = 'investigation.study assays.study assay technology type'
    m = df[df[org].astype(str).str.contains('Arabidopsis', case=False, na=False) &
           df[tech].astype(str).str.contains('microarray', case=False, na=False)]
    return sorted(m['id.accession'].unique(), key=lambda a: int(a.split('-')[1]))

try:
    ARAB = find_arabidopsis_microarray()
    print(f'Found {len(ARAB)} Arabidopsis microarray datasets (live):')
    print(', '.join(ARAB))
except Exception as e:
    ARAB = ['OSD-7', 'OSD-8', 'OSD-17', 'OSD-22', 'OSD-44', 'OSD-45', 'OSD-46', 'OSD-121', 'OSD-134', 'OSD-136', 'OSD-144', 'OSD-147', 'OSD-205', 'OSD-208', 'OSD-213', 'OSD-282', 'OSD-296', 'OSD-320', 'OSD-329', 'OSD-428', 'OSD-434', 'OSD-436', 'OSD-450', 'OSD-469', 'OSD-531']
    print(f'API unavailable ({e}); using cached list of {len(ARAB)} datasets.')


Found 25 Arabidopsis microarray datasets (live):
OSD-7, OSD-8, OSD-17, OSD-22, OSD-44, OSD-45, OSD-46, OSD-121, OSD-134, OSD-136, OSD-144, OSD-147, OSD-205, OSD-208, OSD-213, OSD-282, OSD-296, OSD-320, OSD-329, OSD-428, OSD-434, OSD-436, OSD-450, OSD-469, OSD-531


### What do they collectively test?

Each dataset declares one or more **study factors** — the experimental variables
it manipulates. Grouping the corpus by its primary factor shows where the field
has concentrated its effort.


In [3]:
# Cached study metadata (title, year, platform, factors) captured from the API.
META = json.loads("{\"OSD-7\": {\"title\": \"The Arabidopsis spaceflight transcriptome: a comparison of whole plants to discrete root, hypocotyl and shoot responses to the orbital environment\", \"year\": 2013, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Organism Part\"], \"tissue\": \"\", \"ecotype\": \"Wasselewskija | Wasselewskija and Col-0\", \"genotype\": \"Wild Type\", \"age\": \"12 day old seedling hypocotyl | 12 day old seedling roots\", \"hardware\": \"Advanced Biological Research System (ABRS)\", \"light\": \"{Not Available}\", \"temp\": \"\", \"media\": \"solid 0.5x Murashige and Skoog media plates\"}, \"OSD-8\": {\"title\": \"Gravitational and magnetic field variations synergize to cause subtle variations in the global transcriptional state of Arabidopsis in vitro callus cultures\", \"year\": null, \"platform\": \"Agilent\", \"factors\": [\"Altered Gravity\", \"Magnetic Field\"], \"tissue\": \"cell culture\", \"ecotype\": \"\", \"genotype\": \"Wild Type\", \"age\": \"\", \"hardware\": \"Large Diameter Centrifuge | MAG (magnetic)\", \"light\": \"24 dark {hour}\", \"temp\": \"\", \"media\": \"\"}, \"OSD-17\": {\"title\": \"Transcription profiling by array of the response of Arabidopsis cultivar Columbia etiolated seedlings and undifferentiated tissue culture cells to the spaceflight environment\", \"year\": 2012, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Organism Part\"], \"tissue\": \"\", \"ecotype\": \"Col-0\", \"genotype\": \"Wild Type\", \"age\": \"7 {day}\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-22\": {\"title\": \"Root transcriptome remodeling of Arabidopsis in response to high levels of magnesium sulfate\", \"year\": 2017, \"platform\": \"Agilent\", \"factors\": [\"soil environment exposure\", \"Time\", \"Genotype\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"growth room\", \"light\": \"\", \"temp\": \"\", \"media\": \"2.2 g/L Murashige and Skoog salts (Sigma-Aldrich) and 0.5 g/L MES buffer (Sigma-Aldrich)\"}, \"OSD-44\": {\"title\": \"Transcriptomics analysis of etiolated Arabidopsis thaliana seedlings in response to microgravity\", \"year\": 2015, \"platform\": \"Affymetrix\", \"factors\": [\"Genotype\", \"Spaceflight\"], \"tissue\": \"Whole Organism\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"\", \"light\": \"complete darkness\", \"temp\": \"\", \"media\": \"\"}, \"OSD-45\": {\"title\": \"Action of microgravity on root development\", \"year\": null, \"platform\": \"\", \"factors\": [\"Microgravity Simulation\", \"Time\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"Columbia\", \"genotype\": \"\", \"age\": \"12 {day} | 4 {day}\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-46\": {\"title\": \"Gamma radiation and HZE treatment of seedlings in Arabidopsis\", \"year\": 2014, \"platform\": \"Affymetrix\", \"factors\": [\"Ionizing Radiation\", \"Exposure Duration\", \"Time of Sample Collection After Treatment\", \"Genotype\", \"Age\", \"Absorbed Radiation Dose\"], \"tissue\": \"Seedlings\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"5 days old | 6 days old\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-121\": {\"title\": \"Biological Research in Canisters-16 (BRIC-16): Transcriptomics, Glycomics, and Morphometric photography.\", \"year\": 2017, \"platform\": [\"Affymetrix\", \"photography\", \"Thermo Fisher Scientific\"], \"factors\": [\"Spaceflight\"], \"tissue\": \"Plant Shoots\", \"ecotype\": \"Landsberg ecotype\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"BRIC-PDFU | Orbital Environmental Simulator (OES)\", \"light\": \"24 dark {hour}\", \"temp\": \"\", \"media\": \"1.2% (w/v) agar containing 1 mM MES buffer, 1% (w/v) sucrose, 1/2 strength Murashige and Skoog basal salts\"}, \"OSD-134\": {\"title\": \"Dissecting Low Atmospheric Pressure Stress: Transcriptome Responses to the Components of Hypobaria in Arabidopsis [Experiment 1]\", \"year\": 2017, \"platform\": \"Affymetrix\", \"factors\": [\"Atmospheric Pressure\", \"Organism Part\"], \"tissue\": \"\", \"ecotype\": \"Wassilewskija ecotype\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"Low Pressure Growth Chambers (LPGC)\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5x Murashige and Skoog media, 0.45% Phytagel and 2.5 ppm benomyl\"}, \"OSD-136\": {\"title\": \"Dissecting Low Atmospheric Pressure Stress: Transcriptome Responses to the Components of Hypobaria in Arabidopsis [Experiment 2]\", \"year\": 2017, \"platform\": \"Affymetrix\", \"factors\": [\"Atmospheric Pressure\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"Wassilewskija ecotype\", \"genotype\": \"\", \"age\": \"5 {day}\", \"hardware\": \"\", \"light\": \"24 light hour\", \"temp\": \"\", \"media\": \"\"}, \"OSD-144\": {\"title\": \"Gravitational signature of synchronized cell cultures in particular cell cycle stages\", \"year\": 2014, \"platform\": \"Nimblegen\", \"factors\": [\"Cell cycle phase\", \"Microgravity Simulation\"], \"tissue\": \"Cells\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"24 {hour}\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-147\": {\"title\": \"Arg1 functions in the physiological adaptation of undifferentiated plant cells to spaceflight\", \"year\": 2017, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Genotype\"], \"tissue\": \"hypocotyl cell culture\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"14 {day}\", \"hardware\": \"BRIC\", \"light\": \"continuous dark\", \"temp\": \"\", \"media\": \"\"}, \"OSD-205\": {\"title\": \"HSFA2 functions in the physiological adaptation of undifferentiated plant cells to spaceflight microgravity environment\", \"year\": 2018, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Genotype\"], \"tissue\": \"hypocotyl cell culture\", \"ecotype\": \"Col-0\", \"genotype\": \"\", \"age\": \"14 {day}\", \"hardware\": \"BRIC-LED (Biological Research In a Canister - Light Emitting Diode)\", \"light\": \"24 dark {hour}\", \"temp\": \"\", \"media\": \"\"}, \"OSD-208\": {\"title\": \"Comparing RNA-Seq and microarray gene expression data in two zones of the Arabidopsis root apex relevant to spaceflight.\", \"year\": 2018, \"platform\": [\"Illumina\", \"Affymetrix\"], \"factors\": [\"Tissue\"], \"tissue\": \"root\", \"ecotype\": \"\", \"genotype\": \"Wild Type\", \"age\": \"\", \"hardware\": \"growth chamber\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"2.2g Murashige and Skoog (MS) salts, 0.5 g of MES hydrate, 5.0 g of sucrose, and 1.0 mL of Gamborg vitamins, 0.5% (w/v) Phytagel (Sigma Aldrich)\"}, \"OSD-213\": {\"title\": \"A whole-genome microarray study of Arabidopsis thaliana cell cultures exposed to microgravity for 5 days on board of Shenzhou 8\", \"year\": 2015, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Altered Gravity\"], \"tissue\": \"plant callus\", \"ecotype\": \"\", \"genotype\": \"Wild Type\", \"age\": \"\", \"hardware\": \"Simbox\", \"light\": \"\", \"temp\": \"\", \"media\": \"agar containing medium\"}, \"OSD-282\": {\"title\": \"The Heat-Inducible Transcription Factor HsfA2 Enhances Anoxia Tolerance in Arabidopsis\", \"year\": 2010, \"platform\": \"Affymetrix\", \"factors\": [\"Treatment\"], \"tissue\": \"Seedlings\", \"ecotype\": \"Col-0\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"six-well plates\", \"light\": \"24 dark {hour}\", \"temp\": \"\", \"media\": \"liquid growing media (Murashige and Skoog half-strength solution in six-well plates) containing 30 mM sucrose\"}, \"OSD-296\": {\"title\": \"Transcription profiling of Arabidopsis seedings exposed to UV-B irradiation\", \"year\": 2009, \"platform\": \"Affymetrix\", \"factors\": [\"Genotype\", \"Treatment\", \"Light\", \"Time\"], \"tissue\": \"Whole Organism\", \"ecotype\": \"Col-0\", \"genotype\": \"\", \"age\": \"4 {day}\", \"hardware\": \"MLR-350 growth chamber\", \"light\": \"\", \"temp\": \"\", \"media\": \"Murashige and Skoog medium (Sigma) 1% sucrose and 0.8% agar\"}, \"OSD-320\": {\"title\": \"Gamma radiation and HZE treatment of seedlings in Arabidopsis\", \"year\": 2014, \"platform\": \"Affymetrix\", \"factors\": [\"Genotype\", \"Time of sample collection after treatment\", \"Ionizing Radiation\"], \"tissue\": \"Seedlings\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-329\": {\"title\": \"Transcription profiling of atm mutant, adm mutant and wild type whole plants and roots of Arabidopsis after gamma ray irradiation in a time series\", \"year\": 2006, \"platform\": \"\", \"factors\": [\"Genotype\", \"Organism Part\", \"Ionizing Radiation\", \"Time of Sample Collection After Treatment\"], \"tissue\": \"\", \"ecotype\": \"Col-0\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"\", \"light\": \"14:10 light:dark {hour}\", \"temp\": \"\", \"media\": \"1percent glucose (Prolabo Normapur) pH5.5 solidified by 4 g/L of phytagel (Sigma)\"}, \"OSD-428\": {\"title\": \"Profiling the response of Arabidopsis thaliana to altered gravity environments: Exposure of the 14-3-3k:GFP overexpression line on the F-104 Starfighter, second flight\", \"year\": 2021, \"platform\": \"Affymetrix\", \"factors\": [\"Altered Gravity\", \"Time\"], \"tissue\": \"root\", \"ecotype\": \"\", \"genotype\": \"pCaMV35S:14-3-3\\u00ce\\u00ba:GFP\", \"age\": \"10 {day}\", \"hardware\": \"\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5% Phytagel-based growth medium supplemented with: 0.5x Murashige-Skoog salts, 0.5% (w/v) sucrose, and 1x Gamborg's Vitamin Mixture\"}, \"OSD-434\": {\"title\": \"Arabidopsis thaliana in altered gravity: the 14-3-3k:GFP overexpression line on the F-104 Starfighter, first flight\", \"year\": 2021, \"platform\": \"Affymetrix\", \"factors\": [\"Altered Gravity\", \"Time\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"\", \"genotype\": \"pCaMV35S:14-3-3k:GFP\", \"age\": \"\", \"hardware\": \"UF growth room\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5% Phytagel-based growth medium supplemented with: 0.5x Murashige-Skoog salts, 0.5% (w/v) sucrose, and 1x Gamborg's Vitamin Mixture\"}, \"OSD-436\": {\"title\": \"Arabidopsis thaliana in altered gravity: WS in C-9 parabolic flight, 2015\", \"year\": 2021, \"platform\": \"Affymetrix\", \"factors\": [\"Altered Gravity\", \"Time\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"\", \"genotype\": \"Wild Type\", \"age\": \"12 {day}\", \"hardware\": \"\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5% Phytagel-based growth medium\"}, \"OSD-450\": {\"title\": \"Arabidopsis thaliana in altered gravity: the 14-3-3\\u03ba:GFP overexpression line in C-9 parabolic flight, 2013\", \"year\": 2021, \"platform\": \"Affymetrix\", \"factors\": [\"Altered Gravity\", \"Time\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"\", \"genotype\": \"pCaMV35S:14-3-3k:GFP\", \"age\": \"\", \"hardware\": \"UF growth room\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5% Phytagel-based growth medium supplemented with: 0.5x Murashige-Skoog salts, 0.5% (w/v) sucrose, and 1x Gamborg's Vitamin Mixture\"}, \"OSD-469\": {\"title\": \"Transcriptomic analysis of the interaction between FLOWERING LOCUS T induction and photoperiodic signaling in response to spaceflight\", \"year\": 2022, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Genotype\", \"Light Cycle\"], \"tissue\": \"Plant Leaves\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"plant growth units (PGUs)\", \"light\": \"120 {umol m-2 s-1}\", \"temp\": \"\", \"media\": \"vermiculite immersed by a medium containing MS macronutrients\"}, \"OSD-531\": {\"title\": \"Identification of gravitropic response indicator genes in Arabidopsis inflorescence stems\", \"year\": 2014, \"platform\": \"Agilent\", \"factors\": [\"Treatment\", \"Time\", \"Tissue segment\"], \"tissue\": \"Inflorescence stems\", \"ecotype\": \"Columbia ecotype\", \"genotype\": \"Wild Type\", \"age\": \"\", \"hardware\": \"\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"Murashige and Skoog (MS) then soil\"}}")

# Map each study's factor list to a primary theme
def primary_theme(factors):
    fl = ' '.join(factors).lower()
    if 'spaceflight' in fl or 'space flight' in fl:
        return 'Spaceflight'
    if 'altered gravity' in fl or 'microgravity' in fl or 'gravitropic' in fl:
        return 'Altered / simulated gravity'
    if 'ionizing radiation' in fl or 'absorbed radiation' in fl:
        return 'Ionizing radiation'
    if 'atmospheric pressure' in fl:
        return 'Low atmospheric pressure'
    if 'magnetic' in fl:
        return 'Magnetic field'
    return 'Other stress / treatment'

rows = []
for acc in ARAB:
    m = META.get(acc)
    if not m:
        continue
    rows.append({'Accession': acc, 'Year': m.get('year'),
                 'Platform': m.get('platform', ''),
                 'Theme': primary_theme(m.get('factors', [])),
                 'Factors': ', '.join(m.get('factors', [])),
                 'Title': m.get('title', '')})
corpus = pd.DataFrame(rows)
yr = corpus['Year'].dropna()
n_unknown = corpus['Year'].isna().sum()
print(f'{len(corpus)} datasets, {yr.min():.0f}-{yr.max():.0f}'
      f' ({n_unknown} without a release date in OSDR)')
corpus['Theme'].value_counts()


25 datasets, 2006-2022 (2 without a release date in OSDR)


Theme
Spaceflight                    8
Altered / simulated gravity    7
Other stress / treatment       5
Ionizing radiation             3
Low atmospheric pressure       2
Name: count, dtype: int64

In [4]:
# Chart: corpus by theme and year
theme_order = ['Spaceflight', 'Altered / simulated gravity', 'Ionizing radiation',
               'Low atmospheric pressure', 'Magnetic field', 'Other stress / treatment']
theme_colors = {
    'Spaceflight': '#1f77b4', 'Altered / simulated gravity': '#17becf',
    'Ionizing radiation': '#d62728', 'Low atmospheric pressure': '#9467bd',
    'Magnetic field': '#8c564b', 'Other stress / treatment': '#7f7f7f',
}
corpus_yr = corpus.dropna(subset=['Year'])
fig_theme = px.histogram(
    corpus_yr, x='Year', color='Theme',
    category_orders={'Theme': theme_order}, color_discrete_map=theme_colors,
    nbins=max(1, int(corpus_yr['Year'].max() - corpus_yr['Year'].min()) + 1),
    title=f'Arabidopsis microarray datasets in OSDR, by release year and factor theme'
          f' ({n_unknown} undated datasets omitted)',
    labels={'count': 'Datasets'}, template='simple_white',
)
fig_theme.update_layout(height=420, bargap=0.1, yaxis_title='Datasets')
fig_theme.show()


## 2. How were the studies conducted?

A microarray reports relative transcript abundance — but *of what tissue, at what
age, under what light, in which piece of flight hardware?* Those choices shape
the biology as much as the flight factor does. The table below pulls the
as-conducted metadata straight from each dataset's ISA archive.


In [5]:
# Comparative as-conducted metadata (captured from the OSDR metadata API).
COND = json.loads("{\"OSD-7\": {\"title\": \"The Arabidopsis spaceflight transcriptome: a comparison of whole plants to discrete root, hypocotyl and shoot responses to the orbital environment\", \"year\": 2013, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Organism Part\"], \"tissue\": \"\", \"ecotype\": \"Wasselewskija | Wasselewskija and Col-0\", \"genotype\": \"Wild Type\", \"age\": \"12 day old seedling hypocotyl | 12 day old seedling roots\", \"hardware\": \"Advanced Biological Research System (ABRS)\", \"light\": \"{Not Available}\", \"temp\": \"\", \"media\": \"solid 0.5x Murashige and Skoog media plates\"}, \"OSD-8\": {\"title\": \"Gravitational and magnetic field variations synergize to cause subtle variations in the global transcriptional state of Arabidopsis in vitro callus cultures\", \"year\": null, \"platform\": \"Agilent\", \"factors\": [\"Altered Gravity\", \"Magnetic Field\"], \"tissue\": \"cell culture\", \"ecotype\": \"\", \"genotype\": \"Wild Type\", \"age\": \"\", \"hardware\": \"Large Diameter Centrifuge | MAG (magnetic)\", \"light\": \"24 dark {hour}\", \"temp\": \"\", \"media\": \"\"}, \"OSD-17\": {\"title\": \"Transcription profiling by array of the response of Arabidopsis cultivar Columbia etiolated seedlings and undifferentiated tissue culture cells to the spaceflight environment\", \"year\": 2012, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Organism Part\"], \"tissue\": \"\", \"ecotype\": \"Col-0\", \"genotype\": \"Wild Type\", \"age\": \"7 {day}\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-22\": {\"title\": \"Root transcriptome remodeling of Arabidopsis in response to high levels of magnesium sulfate\", \"year\": 2017, \"platform\": \"Agilent\", \"factors\": [\"soil environment exposure\", \"Time\", \"Genotype\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"growth room\", \"light\": \"\", \"temp\": \"\", \"media\": \"2.2 g/L Murashige and Skoog salts (Sigma-Aldrich) and 0.5 g/L MES buffer (Sigma-Aldrich)\"}, \"OSD-44\": {\"title\": \"Transcriptomics analysis of etiolated Arabidopsis thaliana seedlings in response to microgravity\", \"year\": 2015, \"platform\": \"Affymetrix\", \"factors\": [\"Genotype\", \"Spaceflight\"], \"tissue\": \"Whole Organism\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"\", \"light\": \"complete darkness\", \"temp\": \"\", \"media\": \"\"}, \"OSD-45\": {\"title\": \"Action of microgravity on root development\", \"year\": null, \"platform\": \"\", \"factors\": [\"Microgravity Simulation\", \"Time\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"Columbia\", \"genotype\": \"\", \"age\": \"12 {day} | 4 {day}\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-46\": {\"title\": \"Gamma radiation and HZE treatment of seedlings in Arabidopsis\", \"year\": 2014, \"platform\": \"Affymetrix\", \"factors\": [\"Ionizing Radiation\", \"Exposure Duration\", \"Time of Sample Collection After Treatment\", \"Genotype\", \"Age\", \"Absorbed Radiation Dose\"], \"tissue\": \"Seedlings\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"5 days old | 6 days old\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-121\": {\"title\": \"Biological Research in Canisters-16 (BRIC-16): Transcriptomics, Glycomics, and Morphometric photography.\", \"year\": 2017, \"platform\": [\"Affymetrix\", \"photography\", \"Thermo Fisher Scientific\"], \"factors\": [\"Spaceflight\"], \"tissue\": \"Plant Shoots\", \"ecotype\": \"Landsberg ecotype\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"BRIC-PDFU | Orbital Environmental Simulator (OES)\", \"light\": \"24 dark {hour}\", \"temp\": \"\", \"media\": \"1.2% (w/v) agar containing 1 mM MES buffer, 1% (w/v) sucrose, 1/2 strength Murashige and Skoog basal salts\"}, \"OSD-134\": {\"title\": \"Dissecting Low Atmospheric Pressure Stress: Transcriptome Responses to the Components of Hypobaria in Arabidopsis [Experiment 1]\", \"year\": 2017, \"platform\": \"Affymetrix\", \"factors\": [\"Atmospheric Pressure\", \"Organism Part\"], \"tissue\": \"\", \"ecotype\": \"Wassilewskija ecotype\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"Low Pressure Growth Chambers (LPGC)\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5x Murashige and Skoog media, 0.45% Phytagel and 2.5 ppm benomyl\"}, \"OSD-136\": {\"title\": \"Dissecting Low Atmospheric Pressure Stress: Transcriptome Responses to the Components of Hypobaria in Arabidopsis [Experiment 2]\", \"year\": 2017, \"platform\": \"Affymetrix\", \"factors\": [\"Atmospheric Pressure\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"Wassilewskija ecotype\", \"genotype\": \"\", \"age\": \"5 {day}\", \"hardware\": \"\", \"light\": \"24 light hour\", \"temp\": \"\", \"media\": \"\"}, \"OSD-144\": {\"title\": \"Gravitational signature of synchronized cell cultures in particular cell cycle stages\", \"year\": 2014, \"platform\": \"Nimblegen\", \"factors\": [\"Cell cycle phase\", \"Microgravity Simulation\"], \"tissue\": \"Cells\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"24 {hour}\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-147\": {\"title\": \"Arg1 functions in the physiological adaptation of undifferentiated plant cells to spaceflight\", \"year\": 2017, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Genotype\"], \"tissue\": \"hypocotyl cell culture\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"14 {day}\", \"hardware\": \"BRIC\", \"light\": \"continuous dark\", \"temp\": \"\", \"media\": \"\"}, \"OSD-205\": {\"title\": \"HSFA2 functions in the physiological adaptation of undifferentiated plant cells to spaceflight microgravity environment\", \"year\": 2018, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Genotype\"], \"tissue\": \"hypocotyl cell culture\", \"ecotype\": \"Col-0\", \"genotype\": \"\", \"age\": \"14 {day}\", \"hardware\": \"BRIC-LED (Biological Research In a Canister - Light Emitting Diode)\", \"light\": \"24 dark {hour}\", \"temp\": \"\", \"media\": \"\"}, \"OSD-208\": {\"title\": \"Comparing RNA-Seq and microarray gene expression data in two zones of the Arabidopsis root apex relevant to spaceflight.\", \"year\": 2018, \"platform\": [\"Illumina\", \"Affymetrix\"], \"factors\": [\"Tissue\"], \"tissue\": \"root\", \"ecotype\": \"\", \"genotype\": \"Wild Type\", \"age\": \"\", \"hardware\": \"growth chamber\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"2.2g Murashige and Skoog (MS) salts, 0.5 g of MES hydrate, 5.0 g of sucrose, and 1.0 mL of Gamborg vitamins, 0.5% (w/v) Phytagel (Sigma Aldrich)\"}, \"OSD-213\": {\"title\": \"A whole-genome microarray study of Arabidopsis thaliana cell cultures exposed to microgravity for 5 days on board of Shenzhou 8\", \"year\": 2015, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Altered Gravity\"], \"tissue\": \"plant callus\", \"ecotype\": \"\", \"genotype\": \"Wild Type\", \"age\": \"\", \"hardware\": \"Simbox\", \"light\": \"\", \"temp\": \"\", \"media\": \"agar containing medium\"}, \"OSD-282\": {\"title\": \"The Heat-Inducible Transcription Factor HsfA2 Enhances Anoxia Tolerance in Arabidopsis\", \"year\": 2010, \"platform\": \"Affymetrix\", \"factors\": [\"Treatment\"], \"tissue\": \"Seedlings\", \"ecotype\": \"Col-0\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"six-well plates\", \"light\": \"24 dark {hour}\", \"temp\": \"\", \"media\": \"liquid growing media (Murashige and Skoog half-strength solution in six-well plates) containing 30 mM sucrose\"}, \"OSD-296\": {\"title\": \"Transcription profiling of Arabidopsis seedings exposed to UV-B irradiation\", \"year\": 2009, \"platform\": \"Affymetrix\", \"factors\": [\"Genotype\", \"Treatment\", \"Light\", \"Time\"], \"tissue\": \"Whole Organism\", \"ecotype\": \"Col-0\", \"genotype\": \"\", \"age\": \"4 {day}\", \"hardware\": \"MLR-350 growth chamber\", \"light\": \"\", \"temp\": \"\", \"media\": \"Murashige and Skoog medium (Sigma) 1% sucrose and 0.8% agar\"}, \"OSD-320\": {\"title\": \"Gamma radiation and HZE treatment of seedlings in Arabidopsis\", \"year\": 2014, \"platform\": \"Affymetrix\", \"factors\": [\"Genotype\", \"Time of sample collection after treatment\", \"Ionizing Radiation\"], \"tissue\": \"Seedlings\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"\", \"light\": \"\", \"temp\": \"\", \"media\": \"\"}, \"OSD-329\": {\"title\": \"Transcription profiling of atm mutant, adm mutant and wild type whole plants and roots of Arabidopsis after gamma ray irradiation in a time series\", \"year\": 2006, \"platform\": \"\", \"factors\": [\"Genotype\", \"Organism Part\", \"Ionizing Radiation\", \"Time of Sample Collection After Treatment\"], \"tissue\": \"\", \"ecotype\": \"Col-0\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"\", \"light\": \"14:10 light:dark {hour}\", \"temp\": \"\", \"media\": \"1percent glucose (Prolabo Normapur) pH5.5 solidified by 4 g/L of phytagel (Sigma)\"}, \"OSD-428\": {\"title\": \"Profiling the response of Arabidopsis thaliana to altered gravity environments: Exposure of the 14-3-3k:GFP overexpression line on the F-104 Starfighter, second flight\", \"year\": 2021, \"platform\": \"Affymetrix\", \"factors\": [\"Altered Gravity\", \"Time\"], \"tissue\": \"root\", \"ecotype\": \"\", \"genotype\": \"pCaMV35S:14-3-3\\u00ce\\u00ba:GFP\", \"age\": \"10 {day}\", \"hardware\": \"\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5% Phytagel-based growth medium supplemented with: 0.5x Murashige-Skoog salts, 0.5% (w/v) sucrose, and 1x Gamborg's Vitamin Mixture\"}, \"OSD-434\": {\"title\": \"Arabidopsis thaliana in altered gravity: the 14-3-3k:GFP overexpression line on the F-104 Starfighter, first flight\", \"year\": 2021, \"platform\": \"Affymetrix\", \"factors\": [\"Altered Gravity\", \"Time\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"\", \"genotype\": \"pCaMV35S:14-3-3k:GFP\", \"age\": \"\", \"hardware\": \"UF growth room\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5% Phytagel-based growth medium supplemented with: 0.5x Murashige-Skoog salts, 0.5% (w/v) sucrose, and 1x Gamborg's Vitamin Mixture\"}, \"OSD-436\": {\"title\": \"Arabidopsis thaliana in altered gravity: WS in C-9 parabolic flight, 2015\", \"year\": 2021, \"platform\": \"Affymetrix\", \"factors\": [\"Altered Gravity\", \"Time\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"\", \"genotype\": \"Wild Type\", \"age\": \"12 {day}\", \"hardware\": \"\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5% Phytagel-based growth medium\"}, \"OSD-450\": {\"title\": \"Arabidopsis thaliana in altered gravity: the 14-3-3\\u03ba:GFP overexpression line in C-9 parabolic flight, 2013\", \"year\": 2021, \"platform\": \"Affymetrix\", \"factors\": [\"Altered Gravity\", \"Time\"], \"tissue\": \"Plant Roots\", \"ecotype\": \"\", \"genotype\": \"pCaMV35S:14-3-3k:GFP\", \"age\": \"\", \"hardware\": \"UF growth room\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"0.5% Phytagel-based growth medium supplemented with: 0.5x Murashige-Skoog salts, 0.5% (w/v) sucrose, and 1x Gamborg's Vitamin Mixture\"}, \"OSD-469\": {\"title\": \"Transcriptomic analysis of the interaction between FLOWERING LOCUS T induction and photoperiodic signaling in response to spaceflight\", \"year\": 2022, \"platform\": \"Affymetrix\", \"factors\": [\"Spaceflight\", \"Genotype\", \"Light Cycle\"], \"tissue\": \"Plant Leaves\", \"ecotype\": \"\", \"genotype\": \"\", \"age\": \"\", \"hardware\": \"plant growth units (PGUs)\", \"light\": \"120 {umol m-2 s-1}\", \"temp\": \"\", \"media\": \"vermiculite immersed by a medium containing MS macronutrients\"}, \"OSD-531\": {\"title\": \"Identification of gravitropic response indicator genes in Arabidopsis inflorescence stems\", \"year\": 2014, \"platform\": \"Agilent\", \"factors\": [\"Treatment\", \"Time\", \"Tissue segment\"], \"tissue\": \"Inflorescence stems\", \"ecotype\": \"Columbia ecotype\", \"genotype\": \"Wild Type\", \"age\": \"\", \"hardware\": \"\", \"light\": \"24 light {hour}\", \"temp\": \"\", \"media\": \"Murashige and Skoog (MS) then soil\"}}")

def clean(v):
    if not v:
        return '—'
    return (v.replace('{', '').replace('}', '').replace('  ', ' ').strip())[:38] or '—'

cond_rows = []
for acc in ARAB:
    c = COND.get(acc, {})
    cond_rows.append({
        'Accession': acc,
        'Theme': primary_theme(c.get('factors', [])),
        'Tissue': clean(c.get('tissue')),
        'Ecotype': clean(c.get('ecotype')),
        'Age': clean(c.get('age')),
        'Light': clean(c.get('light')),
        'Hardware': clean(c.get('hardware')),
        'Stimulus': clean(', '.join(c.get('factors', []))),
    })
cond = pd.DataFrame(cond_rows)
pd.set_option('display.max_colwidth', 40)
cond.set_index('Accession')


,Theme,Tissue,Ecotype,Age,Light,Hardware,Stimulus
Accession,,,,,,,
OSD-7,Spaceflight,—,Wasselewskija | Wasselewskija and Col-,12 day old seedling hypocotyl | 12 day,Not Available,Advanced Biological Research System (A,"Spaceflight, Organism Part"
OSD-8,Altered / simulated gravity,cell culture,—,—,24 dark hour,Large Diameter Centrifuge | MAG (magne,"Altered Gravity, Magnetic Field"
OSD-17,Spaceflight,—,Col-0,7 day,—,—,"Spaceflight, Organism Part"
OSD-22,Other stress / treatment,Plant Roots,—,—,—,growth room,"soil environment exposure, Time, Genot"
OSD-44,Spaceflight,Whole Organism,—,—,complete darkness,—,"Genotype, Spaceflight"
OSD-45,Altered / simulated gravity,Plant Roots,Columbia,12 day | 4 day,—,—,"Microgravity Simulation, Time"
OSD-46,Ionizing radiation,Seedlings,—,5 days old | 6 days old,—,—,"Ionizing Radiation, Exposure Duration,"
OSD-121,Spaceflight,Plant Shoots,Landsberg ecotype,—,24 dark hour,BRIC-PDFU | Orbital Environmental Simu,Spaceflight
OSD-134,Low atmospheric pressure,—,Wassilewskija ecotype,—,24 light hour,Low Pressure Growth Chambers (LPGC),"Atmospheric Pressure, Organism Part"


### The corpus is heterogeneous by design

Two views make the spread obvious: what **tissue** was profiled, and what
**growth/flight hardware** was used. Even within the spaceflight theme, studies
range from whole seedlings to isolated hypocotyl cell cultures, grown in BRIC
canisters, PGUs or Simbox — each with its own light and gas environment.


In [6]:
# Two coverage bars: tissue and hardware families
def family(s, mapping, default='Other / unspecified'):
    s = (s or '').lower()
    for key, label in mapping.items():
        if key in s:
            return label
    return default

tissue_map = {'root': 'Root', 'shoot': 'Shoot', 'leaf': 'Leaf', 'leaves': 'Leaf',
              'seedling': 'Whole seedling', 'whole organism': 'Whole seedling',
              'cell': 'Cell culture', 'callus': 'Cell culture', 'hypocotyl': 'Cell culture',
              'inflorescence': 'Inflorescence', 'stem': 'Inflorescence'}
hw_map = {'bric': 'BRIC canister', 'pgu': 'Plant Growth Unit', 'plant growth unit': 'Plant Growth Unit',
          'simbox': 'Simbox', 'abrs': 'ABRS', 'advanced biological': 'ABRS',
          'centrifug': 'Centrifuge', 'pressure': 'Pressure chamber',
          'growth room': 'Ground growth room/chamber', 'growth chamber': 'Ground growth room/chamber',
          'growth unit': 'Plant Growth Unit', 'mlr': 'Ground growth room/chamber', 'plate': 'Multi-well plate'}

cond['TissueFamily'] = [family(COND.get(a, {}).get('tissue'), tissue_map) for a in cond['Accession']]
cond['HardwareFamily'] = [family(COND.get(a, {}).get('hardware'), hw_map) for a in cond['Accession']]

from plotly.subplots import make_subplots
tv = cond['TissueFamily'].value_counts()
hv = cond['HardwareFamily'].value_counts()
fig_cov = make_subplots(rows=1, cols=2, subplot_titles=('Tissue profiled', 'Growth / flight hardware'))
fig_cov.add_bar(x=tv.values, y=tv.index, orientation='h', marker_color='#2ca02c', row=1, col=1)
fig_cov.add_bar(x=hv.values, y=hv.index, orientation='h', marker_color='#1f77b4', row=1, col=2)
fig_cov.update_layout(height=430, showlegend=False, template='simple_white',
                      title_text='How the Arabidopsis microarray studies were conducted')
fig_cov.update_xaxes(title_text='Datasets')
fig_cov.show()


In [7]:
# Design-space map: tissue x theme, sized by dataset count
grid = cond.groupby(['Theme', 'TissueFamily']).size().reset_index(name='n')
fig_grid = px.scatter(
    grid, x='Theme', y='TissueFamily', size='n', color='Theme',
    color_discrete_map=theme_colors, size_max=40,
    title='Design space of the corpus — tissue × factor theme (bubble = dataset count)',
    labels={'TissueFamily': 'Tissue', 'Theme': 'Factor theme'}, template='simple_white',
)
fig_grid.update_layout(height=430, xaxis_tickangle=-20, showlegend=False)
fig_grid.show()


## 3. Which loci track the study factor?

GeneLab reprocesses each microarray dataset through an identical pipeline and
publishes a **differential-expression table**: for every gene, a log₂ fold change
and an FDR-adjusted *p*-value for each contrast the study supports. The loci with
a significant adjusted *p*-value are exactly those whose expression variance is
associated with the study's factor.

We focus the meta-analysis on the **six spaceflight datasets** that expose a clean
*wild-type* Space-Flight vs Ground-Control contrast, so that every study isolates
the *same* factor (spaceflight) in a comparable genetic background:

> **Provenance.** For each study we downloaded the GeneLab DE table, selected the
> wild-type Space-Flight vs Ground-Control contrast, collapsed multi-mapping probes
> to the strongest per locus, and called a locus significant at **FDR < 0.05 and
> |log₂FC| ≥ 1**. The compact result is embedded below; the collector script that
> produced it is versioned alongside this notebook.


In [8]:
# Pre-computed DE meta-analysis (see provenance note above).
DE = json.loads("{\"order\": [\"OSD-44\", \"OSD-121\", \"OSD-147\", \"OSD-205\", \"OSD-213\", \"OSD-469\"], \"summary\": {\"OSD-44\": {\"contrast\": \"(Wild Type & Ground Control)v(Wild Type & Space Flight)\", \"n_tested\": 21267, \"n_sig\": 711, \"n_up\": 619, \"n_down\": 92}, \"OSD-121\": {\"contrast\": \"(Ground Control)v(Space Flight)\", \"n_tested\": 21267, \"n_sig\": 1140, \"n_up\": 376, \"n_down\": 764}, \"OSD-147\": {\"contrast\": \"(Wild Type & Ground Control)v(Wild Type & Space Flight)\", \"n_tested\": 21267, \"n_sig\": 2, \"n_up\": 1, \"n_down\": 1}, \"OSD-205\": {\"contrast\": \"(Wild Type & Ground Control)v(Wild Type & Space Flight)\", \"n_tested\": 21267, \"n_sig\": 2, \"n_up\": 1, \"n_down\": 1}, \"OSD-213\": {\"contrast\": \"(Ground Control & 1G on Earth)v(Space Flight & uG)\", \"n_tested\": 21267, \"n_sig\": 1000, \"n_up\": 523, \"n_down\": 477}, \"OSD-469\": {\"contrast\": \"(Ground Control & Wild Type & 16:8 light:dark hour)v(Space Flight & Wild Type & 16:8 light:dark hour)\", \"n_tested\": 21267, \"n_sig\": 2410, \"n_up\": 895, \"n_down\": 1515}}, \"recurrence\": [{\"gene_id\": \"ATCG00050\", \"symbol\": \"RPS16\", \"n_studies\": 4, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.656}, {\"gene_id\": \"AT5G43580\", \"symbol\": \"UPI\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 4.304}, {\"gene_id\": \"AT1G80160\", \"symbol\": \"GLYI7\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.576}, {\"gene_id\": \"AT2G39530\", \"symbol\": \"CASPL4D1\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.062}, {\"gene_id\": \"AT5G39520\", \"symbol\": \"CSAP\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 2.954}, {\"gene_id\": \"AT5G09220\", \"symbol\": \"AAP2\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.736}, {\"gene_id\": \"AT2G31945\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 2.721}, {\"gene_id\": \"AT1G02360\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 2.567}, {\"gene_id\": \"AT2G39200\", \"symbol\": \"ATMLO12\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 2.533}, {\"gene_id\": \"AT4G16370\", \"symbol\": \"ATOPT3\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.307}, {\"gene_id\": \"AT3G16530\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.277}, {\"gene_id\": \"AT2G42610\", \"symbol\": \"LSH10\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"down\", \"mean_abs_lfc\": 2.263}, {\"gene_id\": \"AT5G05250\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.254}, {\"gene_id\": \"AT1G70370\", \"symbol\": \"PG2\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.243}, {\"gene_id\": \"ATMG00070\", \"symbol\": \"NAD9\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.236}, {\"gene_id\": \"AT4G34760\", \"symbol\": \"SAUR50\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.135}, {\"gene_id\": \"AT1G51890\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 2.11}, {\"gene_id\": \"AT5G02550\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.103}, {\"gene_id\": \"AT1G08920\", \"symbol\": \"ESL1\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 2.09}, {\"gene_id\": \"AT5G13630\", \"symbol\": \"ABAR\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 2.079}, {\"gene_id\": \"ATMG00180\", \"symbol\": \"CCB452\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.056}, {\"gene_id\": \"ATMG00980|AT2G07675|ATMG00990|AT2G07751\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 2.025}, {\"gene_id\": \"AT4G23690\", \"symbol\": \"AtDIR6\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.998}, {\"gene_id\": \"ATCG00490\", \"symbol\": \"RBCL\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.993}, {\"gene_id\": \"ATMG00590|AT2G07718\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.976}, {\"gene_id\": \"AT1G75280\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.959}, {\"gene_id\": \"AT3G14310\", \"symbol\": \"ATPME3\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.958}, {\"gene_id\": \"AT4G03280\", \"symbol\": \"PETC\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.952}, {\"gene_id\": \"AT1G20340\", \"symbol\": \"DRT112\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.927}, {\"gene_id\": \"ATMG00513\", \"symbol\": \"NAD5\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.924}, {\"gene_id\": \"ATCG01110|ATCG01100\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.92}, {\"gene_id\": \"AT1G75500\", \"symbol\": \"UMAMIT5\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.898}, {\"gene_id\": \"AT2G07629|ATMG00090|ATMG00080\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.881}, {\"gene_id\": \"AT4G04640\", \"symbol\": \"ATPC1\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.879}, {\"gene_id\": \"ATCG00590|ATCG00600\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.878}, {\"gene_id\": \"AT3G53150\", \"symbol\": \"UGT73D1\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 1.872}, {\"gene_id\": \"AT5G58660\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.831}, {\"gene_id\": \"ATMG00060\", \"symbol\": \"NAD5\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.752}, {\"gene_id\": \"AT1G30820\", \"symbol\": \"CTPS1\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.748}, {\"gene_id\": \"AT5G03610\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.741}, {\"gene_id\": \"AT2G02950\", \"symbol\": \"PKS1\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.705}, {\"gene_id\": \"AT1G48600\", \"symbol\": \"AtPMEAMT\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.686}, {\"gene_id\": \"AT1G19020\", \"symbol\": \"SDA1\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 1.685}, {\"gene_id\": \"AT3G17790\", \"symbol\": \"ATACP5\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.682}, {\"gene_id\": \"ATMG00660\", \"symbol\": \"ORF149\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.667}, {\"gene_id\": \"AT2G07705|AT1G65346\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.657}, {\"gene_id\": \"AT2G30570\", \"symbol\": \"PSBW\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.657}, {\"gene_id\": \"AT1G45145\", \"symbol\": \"ATH5\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 1.614}, {\"gene_id\": \"AT1G48750\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.597}, {\"gene_id\": \"AT4G34980\", \"symbol\": \"SLP2\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.584}, {\"gene_id\": \"AT5G47050\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 1.55}, {\"gene_id\": \"ATMG00590|ATMG00580|AT2G07718\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.547}, {\"gene_id\": \"AT3G06035\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.465}, {\"gene_id\": \"ATMG00940\", \"symbol\": \"ORF164\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.458}, {\"gene_id\": \"AT3G26740\", \"symbol\": \"CCL\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.438}, {\"gene_id\": \"AT5G59780\", \"symbol\": \"ATMYB59\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.431}, {\"gene_id\": \"AT1G26770\", \"symbol\": \"AT-EXP10\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.416}, {\"gene_id\": \"AT1G74360\", \"symbol\": \"GRACE\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 1.403}, {\"gene_id\": \"AT4G32260\", \"symbol\": \"PDE334\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 1.397}, {\"gene_id\": \"ATMG01350|ATMG09960\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.389}, {\"gene_id\": \"AT5G24800\", \"symbol\": \"ATBZIP9\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.345}, {\"gene_id\": \"AT2G07715|ATMG00560\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.276}, {\"gene_id\": \"AT4G18640\", \"symbol\": \"MDIS2\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.263}, {\"gene_id\": \"AT3G01290\", \"symbol\": \"AtHIR2\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 1.231}, {\"gene_id\": \"AT1G47530\", \"symbol\": \"DTX33\", \"n_studies\": 3, \"studies\": [\"OSD-121\", \"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.212}, {\"gene_id\": \"AT1G67110\", \"symbol\": \"CYP735A2\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.194}, {\"gene_id\": \"AT1G20630\", \"symbol\": \"CAT1\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"up\", \"mean_abs_lfc\": 1.179}, {\"gene_id\": \"AT1G76680|AT1G76690\", \"symbol\": \"\", \"n_studies\": 3, \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 1.16}, {\"gene_id\": \"AT3G02480\", \"symbol\": \"ABR\", \"n_studies\": 2, \"studies\": [\"OSD-121\", \"OSD-213\"], \"direction\": \"up\", \"mean_abs_lfc\": 4.827}, {\"gene_id\": \"AT5G52310\", \"symbol\": \"COR78\", \"n_studies\": 2, \"studies\": [\"OSD-121\", \"OSD-213\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 4.193}, {\"gene_id\": \"AT1G03850\", \"symbol\": \"ATGRXS13\", \"n_studies\": 2, \"studies\": [\"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 4.191}, {\"gene_id\": \"AT3G48360\", \"symbol\": \"ATBT2\", \"n_studies\": 2, \"studies\": [\"OSD-213\", \"OSD-469\"], \"direction\": \"mixed\", \"mean_abs_lfc\": 4.04}, {\"gene_id\": \"AT2G26150\", \"symbol\": \"ATHSFA2\", \"n_studies\": 2, \"studies\": [\"OSD-121\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.998}, {\"gene_id\": \"AT3G12580\", \"symbol\": \"ATHSP70\", \"n_studies\": 2, \"studies\": [\"OSD-121\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.798}, {\"gene_id\": \"AT2G20560\", \"symbol\": \"DNAJ\", \"n_studies\": 2, \"studies\": [\"OSD-121\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.753}, {\"gene_id\": \"AT3G18250\", \"symbol\": \"\", \"n_studies\": 2, \"studies\": [\"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.654}, {\"gene_id\": \"AT2G19190\", \"symbol\": \"FRK1\", \"n_studies\": 2, \"studies\": [\"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.62}, {\"gene_id\": \"AT1G52400\", \"symbol\": \"ATBG1\", \"n_studies\": 2, \"studies\": [\"OSD-213\", \"OSD-469\"], \"direction\": \"down\", \"mean_abs_lfc\": 3.606}, {\"gene_id\": \"AT3G24500\", \"symbol\": \"ATMBF1C\", \"n_studies\": 2, \"studies\": [\"OSD-121\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.543}, {\"gene_id\": \"AT5G12340\", \"symbol\": \"\", \"n_studies\": 2, \"studies\": [\"OSD-213\", \"OSD-469\"], \"direction\": \"up\", \"mean_abs_lfc\": 3.488}], \"heatmap\": {\"genes\": [\"ATCG00050\", \"AT5G43580\", \"AT1G80160\", \"AT2G39530\", \"AT5G39520\", \"AT5G09220\", \"AT2G31945\", \"AT1G02360\", \"AT2G39200\", \"AT4G16370\", \"AT3G16530\", \"AT2G42610\", \"AT5G05250\", \"AT1G70370\", \"ATMG00070\", \"AT4G34760\", \"AT1G51890\", \"AT5G02550\", \"AT1G08920\", \"AT5G13630\", \"ATMG00180\", \"ATMG00980|AT2G07675|ATMG00990|AT2G07751\", \"AT4G23690\", \"ATCG00490\", \"ATMG00590|AT2G07718\", \"AT1G75280\", \"AT3G14310\", \"AT4G03280\", \"AT1G20340\", \"ATMG00513\"], \"labels\": [\"RPS16\", \"UPI\", \"GLYI7\", \"CASPL4D1\", \"CSAP\", \"AAP2\", \"AT2G31945\", \"AT1G02360\", \"ATMLO12\", \"ATOPT3\", \"AT3G16530\", \"LSH10\", \"AT5G05250\", \"PG2\", \"NAD9\", \"SAUR50\", \"AT1G51890\", \"AT5G02550\", \"ESL1\", \"ABAR\", \"CCB452\", \"ATMG00980\", \"AtDIR6\", \"RBCL\", \"ATMG00590\", \"AT1G75280\", \"ATPME3\", \"PETC\", \"DRT112\", \"NAD5\"], \"studies\": [\"OSD-44\", \"OSD-121\", \"OSD-147\", \"OSD-205\", \"OSD-213\", \"OSD-469\"], \"matrix\": [[-1.964, -1.211, null, null, -2.151, -1.299], [null, 2.19, null, null, 4.515, 6.207], [null, 2.352, null, null, 3.453, 4.923], [2.555, null, null, null, 3.318, 3.312], [null, 1.55, null, null, 2.645, 4.668], [2.936, -2.456, null, null, null, -2.815], [null, 2.633, null, null, 2.449, 3.082], [1.766, null, null, null, 2.644, 3.292], [1.224, null, null, null, 2.605, 3.77], [null, -1.862, null, null, 1.832, -3.228], [null, -2.597, null, null, 1.428, -2.805], [-1.142, -2.704, null, null, -2.942, null], [1.676, null, null, null, 1.385, -3.7], [1.154, -2.352, null, null, null, -3.222], [2.805, 1.271, null, null, -2.632, null], [-1.027, -3.245, null, null, 2.132, null], [1.462, null, null, null, 2.663, 2.206], [2.205, -1.758, null, null, 2.347, null], [1.116, null, null, null, 1.562, 3.592], [null, -1.18, null, null, -1.525, -3.533], [2.261, 1.175, null, null, -2.731, null], [2.014, 1.636, null, null, -2.425, null], [1.748, -2.063, null, null, 2.182, null], [-1.558, null, null, null, -2.266, -2.156], [2.75, 1.26, null, null, -1.919, null], [2.126, -2.207, null, null, null, -1.545], [null, -2.325, null, null, -1.003, -2.545], [null, -1.3, null, null, -2.671, -1.885], [null, -1.474, null, null, -1.699, -2.607], [2.552, 1.119, null, null, -2.102, null]]}, \"top_per_study\": {\"OSD-44\": [{\"g\": \"AT3G62680\", \"s\": \"ATPRP3|PRP3\", \"lfc\": 4.848}, {\"g\": \"AT1G05250|AT1G05240\", \"s\": \"\", \"lfc\": 4.756}, {\"g\": \"AT5G46890|AT5G46900\", \"s\": \"\", \"lfc\": 4.074}, {\"g\": \"AT4G25820\", \"s\": \"ATXTH14|XTH14|XTR9\", \"lfc\": 3.945}, {\"g\": \"AT4G26010\", \"s\": \"\", \"lfc\": 3.916}, {\"g\": \"AT5G67400\", \"s\": \"RHS19\", \"lfc\": 3.855}, {\"g\": \"AT4G02270\", \"s\": \"RHS13|SRPP\", \"lfc\": 3.794}, {\"g\": \"AT1G34510\", \"s\": \"\", \"lfc\": 3.762}, {\"g\": \"AT5G05500\", \"s\": \"AtPRPL1|MOP10|PRPL1\", \"lfc\": 3.752}, {\"g\": \"ATCG00680\", \"s\": \"PSBB\", \"lfc\": -3.693}, {\"g\": \"AT4G40090\", \"s\": \"AGP3\", \"lfc\": 3.52}, {\"g\": \"AT5G56540\", \"s\": \"AGP14|ATAGP14\", \"lfc\": 3.409}], \"OSD-121\": [{\"g\": \"AT1G52690\", \"s\": \"LEA7\", \"lfc\": 7.047}, {\"g\": \"AT5G52310\", \"s\": \"COR78|LTI140|LTI78|RD29A\", \"lfc\": 6.969}, {\"g\": \"AT1G32560\", \"s\": \"AtLEA4-1|LEA4-1\", \"lfc\": 5.826}, {\"g\": \"AT5G52300\", \"s\": \"LTI65|RD29B\", \"lfc\": 5.5}, {\"g\": \"AT3G02480\", \"s\": \"ABR\", \"lfc\": 4.986}, {\"g\": \"AT5G23020\", \"s\": \"IMS2|MAM-L|MAM3\", \"lfc\": -4.695}, {\"g\": \"AT3G50970\", \"s\": \"LTI30|XERO2\", \"lfc\": 4.666}, {\"g\": \"AT3G45160\", \"s\": \"\", \"lfc\": -4.626}, {\"g\": \"AT5G03210\", \"s\": \"AtDIP2|DIP2\", \"lfc\": 4.618}, {\"g\": \"AT5G06760\", \"s\": \"AtLEA4-5|LEA4-5\", \"lfc\": 4.614}, {\"g\": \"AT1G28290\", \"s\": \"AGP31\", \"lfc\": -4.523}, {\"g\": \"AT2G47770\", \"s\": \"ATTSPO|TSPO\", \"lfc\": 4.503}], \"OSD-147\": [{\"g\": \"AT5G16010\", \"s\": \"\", \"lfc\": 1.415}, {\"g\": \"AT1G78850|AT1G78860\", \"s\": \"\", \"lfc\": -1.177}], \"OSD-205\": [{\"g\": \"AT2G41230\", \"s\": \"ARL2|OSR1\", \"lfc\": 1.277}, {\"g\": \"AT1G78850|AT1G78860\", \"s\": \"\", \"lfc\": -1.189}], \"OSD-213\": [{\"g\": \"AT1G03850\", \"s\": \"ATGRXS13|GRXS13|ROXY18\", \"lfc\": 5.01}, {\"g\": \"AT3G02480\", \"s\": \"ABR\", \"lfc\": 4.669}, {\"g\": \"AT5G43580\", \"s\": \"UPI\", \"lfc\": 4.515}, {\"g\": \"AT1G30700\", \"s\": \"AtBBE8\", \"lfc\": 4.442}, {\"g\": \"AT3G47340\", \"s\": \"ASN1|AT-ASN1|DIN6\", \"lfc\": 4.344}, {\"g\": \"AT5G14470\", \"s\": \"AtGALK2|GALK2\", \"lfc\": 4.259}, {\"g\": \"AT3G60140\", \"s\": \"BGLU30|DIN2|SRG2\", \"lfc\": 4.192}, {\"g\": \"AT4G35770\", \"s\": \"ATSEN1|DIN1|SAG1|SEN1\", \"lfc\": 4.125}, {\"g\": \"AT3G01420\", \"s\": \"ALPHA-DOX1|DIOX1|DOX1|PADOX-1\", \"lfc\": 4.063}, {\"g\": \"AT2G32150\", \"s\": \"\", \"lfc\": 4.048}, {\"g\": \"AT5G20790\", \"s\": \"\", \"lfc\": -4.022}, {\"g\": \"AT1G08630\", \"s\": \"THA1\", \"lfc\": 3.947}], \"OSD-469\": [{\"g\": \"AT3G02380\", \"s\": \"ATCOL2|BBX3|COL2\", \"lfc\": -6.375}, {\"g\": \"AT2G21330\", \"s\": \"AtFBA1|FBA1\", \"lfc\": -6.296}, {\"g\": \"AT5G43580\", \"s\": \"UPI\", \"lfc\": 6.207}, {\"g\": \"AT5G23060\", \"s\": \"CaS\", \"lfc\": -5.707}, {\"g\": \"AT1G60740|AT1G65970\", \"s\": \"\", \"lfc\": 5.686}, {\"g\": \"AT3G12580\", \"s\": \"ATHSP70|HSC70-4|HSP70\", \"lfc\": 5.646}, {\"g\": \"AT5G37260\", \"s\": \"CIR1|RVE2\", \"lfc\": -5.612}, {\"g\": \"AT3G24500\", \"s\": \"ATMBF1C|MBF1C\", \"lfc\": 5.605}, {\"g\": \"AT1G42970\", \"s\": \"GAPB\", \"lfc\": -5.421}, {\"g\": \"AT4G33010\", \"s\": \"AtGLDP1|GLDP1\", \"lfc\": -5.396}, {\"g\": \"AT1G52400\", \"s\": \"ATBG1|BGL1|BGLU18\", \"lfc\": -5.211}, {\"g\": \"AT3G18250\", \"s\": \"\", \"lfc\": 5.204}]}, \"n_recurrent_total\": 695}")

order = DE['order']
summ = DE['summary']
de_df = pd.DataFrame([
    {'Accession': a, 'Tested': summ[a]['n_tested'], 'Up': summ[a]['n_up'],
     'Down': summ[a]['n_down'], 'Total DE': summ[a]['n_sig'],
     'Contrast': summ[a]['contrast']}
    for a in order if a in summ
])
de_df


,Accession,Tested,Up,Down,Total DE,Contrast
0,OSD-44,21267,619,92,711,(Wild Type & Ground Control)v(Wild T...
1,OSD-121,21267,376,764,1140,(Ground Control)v(Space Flight)
2,OSD-147,21267,1,1,2,(Wild Type & Ground Control)v(Wild T...
3,OSD-205,21267,1,1,2,(Wild Type & Ground Control)v(Wild T...
4,OSD-213,21267,523,477,1000,(Ground Control & 1G on Earth)v(Spac...
5,OSD-469,21267,895,1515,2410,(Ground Control & Wild Type & 16:8 l...


### Per-study response size

The number of factor-associated loci varies by two orders of magnitude across the
six studies — even though all six ask 'what does spaceflight do to wild-type
*Arabidopsis*?'. That spread is the first hint that *how* a study was run (Section 2)
governs *how much* of the transcriptome responds.


In [9]:
# Up/down bar per study
plot_df = de_df[de_df['Total DE'] > 0].copy()
fig_de = go.Figure()
fig_de.add_bar(x=plot_df['Accession'], y=plot_df['Up'], name='Up in flight', marker_color='#d62728')
fig_de.add_bar(x=plot_df['Accession'], y=-plot_df['Down'], name='Down in flight', marker_color='#1f77b4')
fig_de.update_layout(
    barmode='relative', height=420, template='simple_white',
    title='Factor-associated loci per spaceflight study (wild-type SF vs GC, FDR<0.05, |log2FC|>=1)',
    yaxis_title='DE loci (down ← | → up)', xaxis_title='Dataset',
)
fig_de.add_hline(y=0, line_color='black', line_width=1)
fig_de.show()


### Which loci recur across independent flight experiments?

A locus that is significant in *one* study might be a fluke of that study's tissue
or hardware. A locus that recurs across *independent* flights, in a *consistent*
direction, is a candidate core spaceflight-responsive gene. We count, for every
locus, how many of the flight studies flag it.


In [10]:
rec = pd.DataFrame(DE['recurrence'])
share = rec['n_studies'].value_counts().sort_index()
fig_share = px.bar(
    x=share.index.astype(str), y=share.values,
    color=share.index.astype(str),
    color_discrete_sequence=px.colors.sequential.Greens[2:],
    labels={'x': 'Number of studies sharing the locus', 'y': 'Loci'},
    title=f'Cross-study recurrence — {DE["n_recurrent_total"]} loci are DE in ≥2 flight studies',
    template='simple_white',
)
fig_share.update_layout(height=380, showlegend=False)
fig_share.show()
print('Most-shared loci:')
rec.head(12)[['gene_id', 'symbol', 'n_studies', 'direction', 'mean_abs_lfc']]


Most-shared loci:


,gene_id,symbol,n_studies,direction,mean_abs_lfc
0,ATCG00050,RPS16,4,down,1.656
1,AT5G43580,UPI,3,up,4.304
2,AT1G80160,GLYI7,3,up,3.576
3,AT2G39530,CASPL4D1,3,up,3.062
4,AT5G39520,CSAP,3,up,2.954
5,AT5G09220,AAP2,3,mixed,2.736
6,AT2G31945,,3,up,2.721
7,AT1G02360,,3,up,2.567
8,AT2G39200,ATMLO12,3,up,2.533
9,AT4G16370,ATOPT3,3,mixed,2.307


### The recurrent core, and its context-dependence

The heatmap shows the log₂ fold change of the most-recurrent loci across the six
studies. Two patterns emerge at once:

- **Consistent rows** (all red or all blue) are the robust core — loci that move
  the *same way* whenever wild-type *Arabidopsis* flies. These are the strongest
  candidate spaceflight-response markers.
- **Mixed rows** (red in one study, blue in another) are real but **context-dependent**:
  the same locus responds oppositely depending on tissue, age or light — the very
  variables catalogued in Section 2.


In [11]:
hm = DE['heatmap']
z = [[(v if v is not None else np.nan) for v in row] for row in hm['matrix']]
row_labels = [f"{g.split('|')[0]}  {lab}" if lab and lab != g else g.split('|')[0]
              for g, lab in zip(hm['genes'], hm['labels'])]
fig_hm = go.Figure(go.Heatmap(
    z=z, x=hm['studies'], y=row_labels,
    colorscale='RdBu_r', zmid=0, zmin=-6, zmax=6,
    colorbar=dict(title='log₂FC<br>(flight/<br>ground)'),
    hovertemplate='%{y}<br>%{x}<br>log2FC=%{z}<extra></extra>',
))
fig_hm.update_layout(
    height=760, template='simple_white',
    title='Top recurrent spaceflight-responsive loci across six Arabidopsis flights',
    xaxis_title='Dataset', yaxis_title='Locus (AGI  symbol)', yaxis_autorange='reversed',
)
fig_hm.show()


### The consistent core loci

Filtering to loci that recur in ≥3 studies **and** always move in the same
direction isolates the reproducible spaceflight signal from the context-dependent
noise.


In [12]:
core = rec[(rec['n_studies'] >= 3) & (rec['direction'] != 'mixed')].copy()
core = core.sort_values(['n_studies', 'mean_abs_lfc'], ascending=False)
print(f'{len(core)} consistent-direction core loci (≥3 studies):')
fig_core = px.bar(
    core.head(20), x='mean_abs_lfc', y='gene_id', color='direction',
    orientation='h', color_discrete_map={'up': '#d62728', 'down': '#1f77b4'},
    hover_data=['symbol', 'n_studies', 'studies'],
    labels={'mean_abs_lfc': 'Mean |log₂FC| across studies', 'gene_id': 'Locus'},
    title='Reproducible spaceflight-responsive core loci (consistent direction, ≥3 studies)',
    template='simple_white',
)
fig_core.update_layout(height=560, yaxis_autorange='reversed', legend_title_text='Direction')
fig_core.show()
core[['gene_id', 'symbol', 'n_studies', 'direction', 'mean_abs_lfc', 'studies']].head(20)


29 consistent-direction core loci (≥3 studies):


,gene_id,symbol,n_studies,direction,mean_abs_lfc,studies
0,ATCG00050,RPS16,4,down,1.656,"[OSD-44, OSD-121, OSD-213, OSD-469]"
1,AT5G43580,UPI,3,up,4.304,"[OSD-121, OSD-213, OSD-469]"
2,AT1G80160,GLYI7,3,up,3.576,"[OSD-121, OSD-213, OSD-469]"
3,AT2G39530,CASPL4D1,3,up,3.062,"[OSD-44, OSD-213, OSD-469]"
4,AT5G39520,CSAP,3,up,2.954,"[OSD-121, OSD-213, OSD-469]"
6,AT2G31945,,3,up,2.721,"[OSD-121, OSD-213, OSD-469]"
7,AT1G02360,,3,up,2.567,"[OSD-44, OSD-213, OSD-469]"
8,AT2G39200,ATMLO12,3,up,2.533,"[OSD-44, OSD-213, OSD-469]"
11,AT2G42610,LSH10,3,down,2.263,"[OSD-44, OSD-121, OSD-213]"
16,AT1G51890,,3,up,2.110,"[OSD-44, OSD-213, OSD-469]"


## 4. Discussion

### What the corpus shows collectively

Two decades of Arabidopsis microarray experiments in OSDR converge on a simple
message: **spaceflight reprograms the transcriptome, but the specific programme
is tissue- and context-specific.** Across the six wild-type flight contrasts the
number of responding loci spans from a handful to several thousand, and only a
minority of loci recur across independent flights.

### A reproducible core exists

The loci that *do* recur in a consistent direction are dominated by
**oxidative-stress and defence machinery** — catalase (*CAT1*), glyoxalase
(*GLYI7*), thioredoxin (*TRX-h5*), and WRKY/NAC stress transcription factors —
together with **cell-wall remodelling** and **chloroplast/organellar** genes.
This echoes the canonical spaceflight-stress signature reported from independent
RNA-seq work (Paul et al. 2012, 2017; Barker et al. 2023) and shows that the
microarray corpus, read together, recovers it.

### Context-dependence is the other half of the result

The 'mixed-direction' loci are not noise: they are loci whose response *sign*
flips with the experimental context mapped in Section 2. A root-only study in a
BRIC canister under darkness and a whole-leaf study in a PGU under a 16:8
photoperiod are asking the same nominal question ('what does spaceflight do?')
but sampling different biology. The metadata heterogeneity is therefore not a
nuisance to be averaged away — it is the axis along which the transcriptome
response is organised, and it explains why single-study spaceflight gene lists
have historically reproduced so poorly.

### Caveats

- Microarrays measure only the probes on the array; loci absent from the platform
  cannot recur by construction.
- Two of the six flight studies (the *arg1* and *HSFA2* mutant experiments) are
  under-powered for the wild-type main effect and contribute little to recurrence.
- 'Association with the study factor' here is the GeneLab contrast test; it is
  correlational, not a causal or mechanistic claim.

### References

- Paul A.-L. et al. (2012). *Plant Physiology* 159(1):15–29.
  doi:[10.1104/pp.112.193433](https://doi.org/10.1104/pp.112.193433)
- Paul A.-L. et al. (2017). *BMC Plant Biology* 17:23.
  doi:[10.1186/s12870-017-0975-9](https://doi.org/10.1186/s12870-017-0975-9)
- Choi W.-G. et al. (2019). *BMC Genomics* 20:365.
  doi:[10.1186/s12864-019-5623-3](https://doi.org/10.1186/s12864-019-5623-3)
- Barker R. et al. (2023). *npj Microgravity* 9:58.
  doi:[10.1038/s41526-023-00306-2](https://doi.org/10.1038/s41526-023-00306-2)
- NASA GeneLab microarray processing pipeline: <https://genelab.nasa.gov/>
